# CH 담당 에이전트 고도화: A1 거시 · A2 재무 · A4 투자심리

코드는 `ch_agents/` 패키지에 있고, 이 노트북은 실행·확인 화면입니다. 테스트는 `pytest members/CH/tests`로 돌립니다.

| 단계 | 하는 일 |
| --- | --- |
| 1. 데이터 | 관점마다 SQL 한 번으로 묶음을 받아 파이썬이 계산 → 숫자 장부(`M1`, `F3`, `S5`…) |
| 2. 판정 | 코드 규칙이 국면·경고·엇갈림을 판정 (숫자는 키로만) |
| 3. LLM | 장부 키만 써서 JSON 작성 → 코드 검사 → 문제 있으면 1회 수정 요청 |
| 4. 정리 | 그래도 남는 검증 안 된 문장은 빼고, LLM 실패 시 규칙 기반 대체 요약 |

**관점별 고도화 내용**

| 관점 | 내용 |
| --- | --- |
| A1 거시 | 금리·환율·물가 국면 판정, 반도체 수출 추세, 주가와 환율·코스피·미국 금리·SOX 동조성(60·250일), 업종별 전달 경로 |
| A2 재무 | 분기 추세, PER TTM/연간 분리, 자기 PER 밴드 위치, 실제 Peer 비교, FCF·주주환원, 이자보상배율 대용치, 부실 경고, 정합성 점검 |
| A4 투자심리 | 수급 강도(거래대금 대비)·연속 일수, 외국인 지분율 1년 위치, 문서 감성 직접 집계, 검색 관심도, VIX, 수급·가격 엇갈림 |

**준비 (Windows cmd)**

```
cd OpenResearch
pip install -r requirements.txt
```

`.env`에 `DATABASE_URL`, `UPSTAGE_API_KEY`가 있어야 합니다. 노트북은 `members/CH` 폴더에서 엽니다.

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import Markdown, display
import pandas as pd

from ch_agents import fetch_agent_data, review_report, run_agent, run_my_agents
from ch_agents.agents import today_kst
from ch_agents.evaluate import cross_check, data_scorecard, kospi200_tickers, llm_scorecard

TICKER = "005930"      # 분석 종목
MODE = "expert"        # expert | guide
AS_OF = today_kst()    # 기준일. 과거 시점 재현은 "2026-10-26"처럼 지정

## 1. 데이터만 확인 (LLM 호출 없음)

Solar를 부르기 전에 장부·판정·무자료를 확인합니다. 숫자가 이상하면 여기서 멈추고 SQL·계산을 고칩니다.

In [ ]:
DATA = {a: fetch_agent_data(a, TICKER, AS_OF) for a in (1, 2, 4)}
for a, d in DATA.items():
    f = d.facts
    md = [f"### A{a} · {d.period}", "", f.table_md(), "", "**코드 판정**", ""] + [f"- {x}" for x in f.flags or ["(없음)"]]
    md += ["", "**해석 가이드**", ""] + [f"- {x}" for x in f.notes or ["(없음)"]]
    md += ["", "**무자료**", ""] + [f"- {g['label']}: {g['reason']}" for g in f.gaps]
    display(Markdown("\n".join(md)))

## 2. 교차 검증

파이썬 계산값 몇 개를 SQL로 따로 다시 계산해 비교합니다. '불일치'가 나오면 계산 로직을 의심합니다.

In [ ]:
pd.DataFrame(cross_check(TICKER, AS_OF))

## 3. 에이전트 실행

위에서 받은 `DATA`를 그대로 넘겨 DB를 다시 조회하지 않습니다. 상태는 다음 중 하나입니다.

- `완료`: 검사 통과
- `완료 (수정 1회)`: 처음 출력에 문제가 있어 한 번 고쳐 통과
- `완료 (일부 문장 제외)`: 고친 뒤에도 남은 검증 안 된 문장을 뺌
- `규칙 기반 대체`: LLM 실패 또는 남은 주장이 없어 코드 판정으로 요약

In [ ]:
res = run_agent(1, data=DATA[1], mode=MODE)
print(res["status"], f"{res['seconds']}초")
display(Markdown(res["analysis"]))

In [ ]:
res = run_agent(2, data=DATA[2], mode=MODE)
print(res["status"], f"{res['seconds']}초")
display(Markdown(res["analysis"]))

In [ ]:
res = run_agent(4, data=DATA[4], mode=MODE)
print(res["status"], f"{res['seconds']}초")
display(Markdown(res["analysis"]))

## 4. 오케스트레이터에 넘길 묶음

키 구성: `agent_id`, `agent_name`, `status`, `period`, `as_of`, `result`(핵심진단·주장·긍정·리스크·미확인·다음점검·점수),
`facts`, `gaps`, `flags`, `checks`, `removed`, `attempts`, `seconds`, `analysis`(마크다운), `source`.
오케스트레이터 담당 팀원에게 이 형식을 공유하면 됩니다.

In [ ]:
MY_OUTPUTS = [run_agent(a, data=DATA[a], mode=MODE) for a in (1, 2, 4)]
pd.DataFrame([{"관점": f"A{o['agent_id']}", "상태": o["status"], "주장": len(o["result"].get("주장", [])),
               "숫자": len(o["facts"]), "무자료": len(o["gaps"]), "판정": len(o["flags"]),
               "제외 문장": len(o["removed"]), "초": o["seconds"]} for o in MY_OUTPUTS])

## 5. 보고서 첨삭 모드

코드가 보고서 숫자를 장부와 대조하고(일치·불일치·대조 불가), 관점마다 빠진 논점과 수정 제안을 냅니다.

In [ ]:
report_text = """
(여기에 첨삭할 보고서를 붙여 넣으세요)
"""
# report_text = open("my_report.md", encoding="utf-8").read()

review = review_report(report_text, TICKER, mode=MODE, datas=[DATA[1], DATA[2], DATA[4]])
display(Markdown("### 수치 대조\n\n" + review["number_check_md"]))
for r in review["reviews"]:
    display(Markdown(f"### A{r['agent_id']} 첨삭\n\n" + r["analysis"]))

## 6. 여러 종목 일괄 점검

데이터 단계(LLM 없음)는 KOSPI 200 전체도 돌릴 수 있습니다. LLM 단계는 비용이 들어 몇 종목만 돌립니다.
결과는 `eval_results.json`에 저장됩니다.

In [ ]:
score = pd.DataFrame(data_scorecard(kospi200_tickers(30), as_of=AS_OF))
score.pivot_table(index=["ticker", "name"], columns="agent", values=["facts", "gaps"], aggfunc="first")

In [ ]:
llm_scorecard(["005930", "000660", "035420"], mode=MODE)

## 7. 정답값 대조표 (삼성전자, `AS_OF = "2026-10-26"`)

2026-10-26 저녁에 같은 정의로 SQL을 따로 돌려 구한 값입니다. `AS_OF = "2026-10-26"`으로 1번 셀을 돌린 결과와 비교하세요.
그 뒤 수집이 더 되면(예: 10월 26일 수급 추가) 값이 조금 달라질 수 있습니다.

| 항목 | 정답값 |
| --- | --- |
| 주가-코스피 상관(250일) | +0.94 |
| 주가-코스피 베타(250일) | +1.30 |
| 주가-원/달러 상관(250일) | -0.01 |
| 외국인 20일 순매수 수량 | -21,805,837주 |
| 외국인 20일 순매수 강도(거래대금 대비) | -5.89% |
| 외국인 연속 순매도 일수 | 2일 |
| 외국인 지분율 20일 변화 | -0.44%p |
| 뉴스 감성(최근 7일) | -0.004 (문서 133건) |
| 뉴스 감성 변화(최근 7일 − 직전 8~30일) | -0.184 → '뉴스 논조 악화' 판정 |
| 검색 관심도 배율(최근 7일 ÷ 직전 90일) | 1.57배 |
| PER 밴드 현재값 | 8.36배 (v_ttm PER(TTM)과 같음) |
| 외국인 순매수일 평균 가격(60일, 근사) | 260,349원 (순매수 22일) |
| 현재가의 외국인 순매수 평균 가격 대비 | -17.23% (종가 215,500원) |